<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 2 — Mappings, service levels and implementation foundations

<small>Data Creators › Day 2 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 2's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day2-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day2-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 2 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

## Today

Yesterday's contract is today's input. By the end of the day it has mapping rules and measurable commitments, you have built a container, and you have generated and run dbt and Airflow code from it.

| # | Section | What happens |
|---|---|---|
| 1 | **[Mappings and service commitments](#s1)** | Demo: STTM and service levels · Activity 6: extend the contract · Break · Shareback: review and present a contract |
| 2 | **[Platform architecture](#s2)** | Demo: warehouse, lake, lakehouse · Demo: Databricks under the hood, Delta, Spark · Demo: how the tools fit together · Shareback · Break |
| 3 | **[Docker](#s3)** | Demo: the five steps and the dev container's Dockerfile · Challenge 7: Docker practical · Lunch |
| 4 | **[Contract to code](#s4)** | Demo: where output lands, live generation · Activity 8: generate and investigate · Break · Debrief · Activity 8b: review another pair's code · Close |

<a id="s1"></a>

## 1. Mappings and service commitments

### Welcome — yesterday's contract is today's input

### Demo — STTM and service levels

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">What is the core of a source-to-target mapping?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>The transformation rules: one per target column, plus the row rules, in English precise enough to implement.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>The STTM is where the product's business logic lives; a vague rule becomes a wrong join nobody notices.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Name the rule type for every column, then the edge cases: nulls, zeros, duplicates, dates, time zones.</div></div></div>

![Sources, rules and target for the running example](../common/diagrams/png/d2-sttm.png)

- Direct, constant, lookup, aggregate, derive: the textbook five; Catalyst splits derive further.
- Row rules matter as much as column rules: dedup, filter, reject, and what a null means in each column.
- The running example's mapping is in common/running-example/sttm.md: eleven column rules and four row rules.

**🖊️ Whiteboard — STTM**

- What is a source-to-target mapping? What is its core?
- What kinds of rule can a column have?
- What is in the design documentation set?

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Delay is allowed to be an hour. Is that the indicator, the objective or the agreement?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>SLI: the measure. SLO: its ideal value. SLA: the permissible departure, agreed, with a consequence.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Consumers hold you to commitments, not to good intentions; each commitment has a cost.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>List the indicators a product needs, give each a target and a tolerance, and say how it is measured.</div></div></div>

![Measure it, target it, promise it with a tolerance](../common/diagrams/png/d2-service-levels.png)

- The indicator is a number you can measure every run; the objective is zero delay, 100% complete; the agreement is what you actually sign.
- Delivery time needs a time zone, the eligible days, the measurement point and the action on failure.
- Duty to explain: when a number looks wrong, somebody owes the consumer an answer within an agreed time.

**🖊️ Whiteboard — service levels**

- What do SLI, SLO and SLA stand for, and what do they mean?
- Give me indicators.
- What is availability, and what does 99.9% mean?
- How do providers get more nines?

| Availability | Downtime a year | Remember it as |
|---|---|---|
| 90% | about 36 days | a month |
| 99% | about 3.65 days | a working week |
| 99.9% | about 8.8 hours | a working day |
| 99.99% | about 53 minutes | an hour |
| 99.999% | about 5 minutes | a coffee |

### Activity 6 — extend the contract

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 6 &nbsp;—&nbsp; Extend the contract &nbsp;·&nbsp; Individual, then pairs or tables &nbsp;·&nbsp; 25 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>On your own, ask Catalyst what source-to-target transformation types and what service-level indicators and objectives it knows. Then, in pairs or at your table, take one of yesterday's data contracts: name the incoming tables and the outgoing table, write the rules that get you from one to the other (lookups, derivations, filters, what nulls mean), and the commitments a consumer would hold you to. Transcribe the discussion, give it to Catalyst, extend the contract, then inspect and revise.</strong><br><br><strong>Produce:</strong><br>• At least one unambiguous mapping rule someone else could implement without asking you<br>• At least one measurable commitment with its indicator, target, tolerance and time zone<br>• The revised contract, shared for review<br><br><strong>Finished early?</strong> Add the breach response: who is told, by when, what they are told.<br><br><strong>Debrief questions:</strong><br>• Could someone else implement this rule without asking you what it means?<br>• For 'available by 07:30': which time zone, which days, measured where, and what happens on failure?<br>• For a lookup: what is the join key, and what happens with zero or several matches?</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">What types of source-to-target transformation do you distinguish, and what service-level indicators and objectives should a data contract state?

/ai-catalyst data-contract
Extend the attached contract with an STTM section (one rule per target column, plus row rules) and measurable SLAs from this discussion transcript. Keep every other section as it is.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">Same questions, then: /data-contract  Extend ... (attach common/running-example/data-contract.md and the discussion)</pre></div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Shareback — review and present a contract

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Markdown editor, then a self-contained HTML page</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Read Markdown properly, then turn the contract into one HTML page on a small model and on a big one.</div></div>

1. Right-click a submitted contract, Open with…, Markdown editor (or Open Preview); edit one cell.
2. In Claude Code (or Copilot) run the prompt below on a fast model at low effort.
3. Open the result in the browser: contents, overview, diagrams.
4. Start the same prompt on the biggest model at high effort; let it run while you teach; compare when it finishes.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, contract attached</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">Create a well-structured, self-contained HTML document from this contract: an overview at the top, a table of contents, diagrams where they help (sources to target, the SLA timeline), and plain-English prose explaining any section a non-engineer would find hard. One file, no external dependencies.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">Same prompt. Fallback already rendered: common/running-example/data-contract.html</pre></div></div>

In [ ]:
# Fallback: the contract already rendered as HTML (made with pandoc; the AI version adds prose and diagrams)
import webbrowser
webbrowser.open((COMMON / "running-example" / "data-contract.html").as_uri())
print("opened", COMMON / "running-example" / "data-contract.html")

### Could someone else implement this rule without asking you what it means?

Take two rules from the room's contracts and test them against this list.

| Ambiguity | Ask | Running-example answer |
|---|---|---|
| Identifiers and keys | Which column joins, and is it unique on the lookup side? | ISIN, one primary row per instrument in the master |
| Null | What does empty mean in this column? | A null FX rate means no same-day rate; the USD price is null and the build fails |
| Zero, minimum, maximum | Is zero a value or a missing value? What is implausible? | A zero or negative price is a quality failure |
| Duplicates | Can the source send the same row twice? Which wins? | Re-deliveries: latest vendor timestamp wins |
| Dates and time zones | Business date or delivery time? Which zone? | Business date of the close; 07:30 America/New_York |
| Scope words | 'US', 'active', 'price': whose definition? | Market and active flag from the master; price is the primary-listing close |

<a id="s2"></a>

## 2. Platform architecture

### Demo — warehouse, lake and lakehouse; Databricks under the hood; Delta; Spark; the tools together

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Why is Databricks on the paved path, and what does it actually do with your dbt SQL?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A lakehouse: Delta tables in S3 (files plus a transaction log), Spark compute run by Databricks, Unity Catalog on top.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Every dbt model you build runs here; knowing what happens under the hood explains the costs, the guarantees and the limits.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Three pictures (warehouse, lake, lakehouse; Databricks under the hood; the Delta format), two demos on this laptop, then how all the tools fit together.</div></div></div>

![Data warehouse, data lake, lakehouse](../common/diagrams/png/d2-warehouse-lake-lakehouse.png)

- A warehouse gives tables, SQL and ACID on its own storage: strong guarantees, expensive to scale, structured data only.
- A lake is files in object storage: cheap and flexible, no guarantees; left alone it becomes a swamp.
- A lakehouse keeps the cheap files and adds an open table format (Delta) for ACID and versions, an engine (Spark) for SQL, and governance (Unity Catalog).

**🖊️ Whiteboard — warehouse, lake, lakehouse**

- What does a warehouse give you, and what does it cost?
- What is a data lake?
- What goes wrong with lakes?
- What does a lakehouse add?

![Databricks under the hood: sources, ingestion, Delta storage, Spark compute, Unity Catalog, consumers](../common/diagrams/png/d2-databricks-under-the-hood.png)

- Loading from different sources: files dropped in S3 (Auto Loader, COPY INTO), databases, streams and APIs (Kafka, Lakeflow), and in training dbt seed.
- Storage is your S3 bucket in Delta format; Databricks does not keep your data inside itself.
- Compute is Spark: clusters for notebooks and jobs, SQL warehouses (Photon) for dbt and BI. Serverless by default on the paved path.
- Unity Catalog names everything catalog.schema.table, holds permissions and lineage, and is what the Marketplace and Catalyst search.

![A Delta table: Parquet files plus a transaction log](../common/diagrams/png/d2-delta-format.png)

- The log is the trick: every write is a commit that lists which files make up the table now.
- That gives ACID, versions and time travel, MERGE and DELETE on files, and schema enforcement.
- OPTIMIZE compacts small files and ZORDER co-locates rows by the columns queries filter on; that is the post-hook in the learners' project.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Delta under the hood, on this laptop</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">A Delta table is Parquet files plus a log: the cell writes one and reads every version back.</div></div>

1. Run the cell: it writes the vendor prices as a Delta table, appends a day, merges a correction, reads version 0, compacts, and reads the table from DuckDB.
2. Open the table folder the script names: the Parquet files and `_delta_log/*.json`.

In [ ]:
!python demos/delta-under-the-hood/demo.py

In [ ]:
!ls -R demos/delta-under-the-hood/warehouse/prices_delta | head -20
!python -c "import json,glob; [print(json.dumps(json.loads(l))[:120]) for l in open(sorted(glob.glob("demos/delta-under-the-hood/warehouse/prices_delta/_delta_log/*.json"))[0]).read().splitlines()[:3]]"
# The log is readable JSON: a commit lists the files it added or removed.

![The Spark engine: a driver plans, executors run the plan over partitions, results combine](../common/diagrams/png/d2-spark-engine.png)

- Spark splits data into partitions and runs the same work on each in parallel across machines; a shuffle (Exchange) regroups rows between stages.
- You write SQL; the driver builds a plan; Databricks provides the machines, the scheduling and the Delta integration.
- dbt never runs your data through its own process: it sends SQL to Databricks and Spark does the work next to the data.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the Spark engine, locally</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">A local Spark shows the driver, executors, partitions and the shuffle that Databricks runs for you at scale.</div></div>

1. Run the cell (about thirty seconds): a local Spark with four executors loads the vendor prices multiplied up and runs a SQL aggregate.
2. Find *Exchange* in the printed physical plan: that is the shuffle between stages.
3. Run it again with `--hold 90` and open http://localhost:4040: Jobs, Stages, the SQL tab.

In [ ]:
!python demos/spark-engine/demo.py 2>&1 | grep -vE 'WARN|^Setting|^To adjust'

![The history in one line: files, databases, files again, and the lakehouse](../common/diagrams/png/d2-data-systems.png)

- Relational databases (1970) gave tables and ACID; the internet pushed data back onto many machines and files; Hadoop then Spark queried the files.
- ACID is linearisability: write 1, 2, 3 and read 1, 2, 3 whatever fails in between; logs make it possible, and Delta's log brings it to files.

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">How do all the tools fit together?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Catalyst generates; the dev container runs dbt and Airflow locally; GitHub holds the code; Harness deploys on merge; Airflow on Astro schedules; Cosmos turns the dbt project into tasks; dbt sends SQL to Databricks; Spark writes Delta on S3; Unity Catalog governs; the Marketplace publishes; Datadog watches.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Every day this week zooms into one or two boxes; the picture is what connects them.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Elicit the role of each tool, then the diagram, then one scheduled run as a sequence.</div></div></div>

![How the tools fit together: generate, develop, ship, schedule, transform, store, govern, find, observe](../common/diagrams/png/d2-tools-together.png)

- Left to right is the life of a change: generated with Catalyst, developed in the dev container, merged on GitHub, deployed by Harness.
- Top to bottom at run time: Airflow decides when, Cosmos maps the dbt project to tasks, dbt sends SQL, Databricks runs Spark, Delta is written to S3.
- Unity Catalog and the Marketplace are where consumers find the product; Datadog is where operations sees it run.

![One scheduled run, as a sequence: scheduler, Cosmos, dbt, Databricks, Delta, Datadog](../common/diagrams/png/d2-scheduled-run.png)

- At 06:00 the scheduler decides a run is due; each Cosmos task runs one dbt model; dbt sends compiled SQL; Spark reads inputs and commits a new Delta version.
- Status flows back up; a failure is an event in Datadog; the publish step is what consumers see.

**🖊️ Whiteboard — the paved path, tool by tool**

- Which technologies are on the paved path?
- What does dbt provide? Databricks? S3?
- Airflow? Astronomer? Cosmos?
- Docker? Git? Harness? Datadog?
- When may you leave the path?

### Shareback — one minute per table

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — one minute per table</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Each table answers in one minute.</div></div>

- One thing Databricks does that a plain file store cannot.
- One thing dbt relies on Databricks for.
- One tool on the diagram you have used, and one you have not.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 11:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s3"></a>

## 3. Docker

### Demo — containers, the challenge's five steps, the dev container's Dockerfile

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Why is your VS Code terminal not your laptop's terminal this week?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A container bundles an application with its dependencies down to the OS user space; an image is the frozen template; a Dockerfile is the recipe.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Repeatable environments, isolation (network, ports, resources), the same bytes on a laptop, a build server and the cloud.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Run an image, look inside, write a four-line Dockerfile, build, run, tag; then the dev container's Dockerfile.</div></div></div>

![Dockerfile to image to container; the dev container is a container VS Code attaches to](../common/diagrams/png/d2-docker.png)

- FROM extends a base image: almost every Dockerfile starts that way; the training container extends an EDO image.
- An image is layered and cached: change the app and only COPY onwards rebuilds.
- A container lives as long as its command; a fresh container starts clean from the image.
- Isolation is the production point: one app per container, controlled ports, capped CPU and memory, a limited blast radius.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the five steps of Challenge 7, in cells</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The challenge's own six-holding app (`demos/docker/portfolio-summary/`), run and built in cells before learners do it.</div></div>

1. Check Docker Desktop is running: the whale in the menu bar, or `docker info`.
2. Cell 1: run `python:3.11-slim` and print its Python; the container exits when the command does.
3. Cell 2: write a file inside one container and list it from another: its own filesystem, gone with the container.
4. Cell 3: read the app's Dockerfile: FROM, WORKDIR, COPY, CMD.
5. Cell 4: build twice; the second build is all CACHED.
6. Cell 5: run the app, then run it with a mounted holdings file: same image, different data.
7. Cell 6: tag the image: two names, one ID; that is how CG ships Airflow and dbt images.

In [ ]:
!docker run --rm python:3.11-slim python --version
# Python from inside a container built from an image, not from this laptop. The container exited when the command did.

In [ ]:
!docker run --rm python:3.11-slim bash -c "head -1 /etc/os-release && touch /tmp/proof.txt && ls /tmp"
!docker run --rm python:3.11-slim ls /tmp
# Its own Linux filesystem; the file written in the first container is gone in the second.

In [ ]:
!grep -E "^(FROM|WORKDIR|COPY|CMD)" demos/docker/portfolio-summary/Dockerfile
# The whole recipe: base image, where, what, how to run.

In [ ]:
!cd demos/docker/portfolio-summary && docker build -t portfolio-summary . 2>&1 | tail -3
!cd demos/docker/portfolio-summary && docker build -t portfolio-summary . 2>&1 | grep -cE 'CACHED' 
# Build twice: the second build is almost entirely cached.

In [ ]:
!docker run --rm portfolio-summary
!docker run --rm -v "$PWD/demos/docker/portfolio-summary/app/holdings.csv:/app/holdings.csv:ro" portfolio-summary
# The app, Python and the OS libraries travel together. Mount a different CSV and the same image gives a different answer.

In [ ]:
!docker tag portfolio-summary portfolio-summary:v1 && docker images portfolio-summary
# Two names, one image ID. Tag, push to a registry, pull elsewhere: that is how CG ships Airflow and dbt images.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the dev container's Dockerfile</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The learners' dev container is a Dockerfile they can now read.</div></div>

1. Open `reference/DataCreators-client/dev-container/edo-training-data-products/.devcontainer/Dockerfile`, the file VS Code builds for learners (the cell below prints its head).
2. Open Docker Desktop: Images, then Settings and Resources, briefly.

In [ ]:
!sed -n "1,12p" ../../reference/DataCreators-client/dev-container/edo-training-data-products/.devcontainer/Dockerfile

### Challenge 7 — Docker practical

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 7 &nbsp;—&nbsp; Docker practical &nbsp;·&nbsp; Individual or pairs &nbsp;·&nbsp; 35 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>In the dev container, work through `challenges/v1/01-docker-intro/README.md` sections 1 to 5: run an image, look inside, write the Dockerfile, build the image, run your image. Section 6 and the 'try a variation' steps are for anyone with time.</strong><br><br><strong>Produce:</strong><br>• `docker run --rm portfolio-summary` printing the six-holding summary<br>• `mise run challenge:verify 01` showing PASS<br>• An answer to: why did the container stop, and what survives when you run a fresh one?<br><br><strong>Finished early?</strong> Multi-stage builds, resource limits (`--memory`, `--cpus`), or read the training repository's own Dockerfile and explain three lines of it.<br><br><strong>Debrief questions:</strong><br>• What is the difference between the image, the running process and its writable files?<br>• What travels with the image, and what does not?</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; Docker practical &nbsp;·&nbsp; 35 min</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Sections 1 to 5: run an image, look inside, write the Dockerfile, build it, run it. &nbsp;·&nbsp; <code>challenges/v1/01-docker-intro/README.md</code></div></div>

**What learners do**

1. `docker --version`; if the engine is not running, start Docker Desktop.
2. Section 1, run an image: a container from a public Python image.
3. Section 2, look inside: the container's own filesystem and processes.
4. Section 3, write the Dockerfile: `starter/Dockerfile` has one TODO per line.
5. Section 4, build the image.
6. Section 5, run the image: the app runs the same anywhere.

**Gate.** `mise run challenge:verify 01` prints PASS.

**Support.**

- Engine not running means Docker Desktop is not started on the laptop.
- CRLF line endings on Windows cause `exec format error`.
- The first pull is slow once.

**Finished early?** Section 6: tags, the variations, multi-stage builds, resource limits.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s4"></a>

## 4. Contract to code

### Demo — where output lands, and one minute of live generation

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">How much of the code can a reviewed contract produce?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A dbt project and an Airflow DAG generated from the PRD, contract, mapping and sample records, then explained and run.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>If most of the code comes from the contract, the interview and the contract were most of the job. The rest is review.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Brief the challenge; show where output lands; generate for one minute live; open the prepared project; then let pairs go.</div></div></div>

![Contract to code: what Catalyst produces and what you still do](../common/diagrams/png/d2-contract-to-code.png)

- setup-dbt-project, write-dbt-model and generate-dag are the Catalyst skills for this; Copilot without a skill also works, with more review.
- The schedule comes from the SLA; the freshness gate from the freshness commitment; the tests from the quality rules.
- You read every line, find the assumption the contract did not state, and fix the contract before the code.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; where output lands: Databricks for learners, DuckDB here</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Learners' tables land in Databricks under their own schema; here the same tables land in a DuckDB file.</div></div>

1. Learners: `mise run setup:databricks` (browser OAuth; Enter through the prompts), then `dpc-check`.
2. Databricks UI: Catalog, `training_dev`, `training_<username>`; every dbt table appears there.
3. Here: run the cell below; it lists the same schema in the DuckDB file once the prepared project is built.

In [ ]:
!bash demos/contract-to-code/eod_prices/run.sh 2>&1 | grep -E 'Done\.|Completed|ERROR' | tail -4
# The running example's contract, implemented: seeds, freshness, build, tests.

In [ ]:
sql("select table_schema, table_name, table_type from information_schema.tables where table_schema = 'training_demo' order by table_type, table_name", HERE / "demos/contract-to-code/eod_prices/warehouse/eod_prices.duckdb")
# What a learner sees in training_dev.training_<user>: the staging views, the intermediate view, the mart table, the seeds.

In [ ]:
sql("select ticker, price_date, price_native, native_ccy, fx_rate_to_usd, price_usd, price_source from training_demo.rpt_eod_prices_us order by price_date desc, ticker limit 8", HERE / "demos/contract-to-code/eod_prices/warehouse/eod_prices.duckdb")

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; one minute of live generation</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Start the generation live, stop after a minute, then open the finished project.</div></div>

1. In Claude Code: `/setup-dbt-project` with `common/running-example/data-contract.md` and `sttm.md` attached.
2. Stop it and open `demos/contract-to-code/eod_prices/models/`: every CTE comments the STTM rule it implements.
3. Open `tests/` and `schema.yml` there: the tests mirror the contract's quality rules.
4. Open `eod_prices_daily.py` in `../day4-airflow/demos/airflow_local/dags/eod_prices/dags/`: the schedule comes from the SLA.

In [ ]:
!sed -n "1,40p" demos/contract-to-code/eod_prices/models/intermediate/int_prices_usd.sql

### Activity 8 — generate and investigate

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 8 &nbsp;—&nbsp; Generate and investigate &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 50 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Give Catalyst your reviewed PRD, contract, mapping and a few sample records; resolve any conflicts between them first. Ask for a dbt project. If you have no source data, ask for simulated data that matches the contract. Add explanatory comments and get instructions for running it and finding the output. Then ask for Airflow orchestration matching the contract's cadence or trigger, and for an explanation of the code and how to run it. Run what you can; inspect the results; prepare one discovery and one correction for the debrief.</strong><br><br><strong>Produce:</strong><br>• A dbt project generated from the contract, with comments<br>• An Airflow DAG matching the contract's schedule<br>• Where the output landed (the schema in Databricks), or why it did not<br>• One discovery and one correction<br><br><strong>Finished early?</strong> Choose one: how is a missing input handled? can a lookup duplicate rows? what happens when the same input is processed twice? Propose the expected result and a check; report it in the debrief. Experienced: investigate the bitemporal macro, or logging from Airflow to Datadog.<br><br><strong>Debrief questions:</strong><br>• Which design choice did the generator make for you, and would you keep it?<br>• Which edge case did the contract not mention?<br>• What did you correct: the specification, or the implementation?</div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, contract + mapping + a few sample rows attached</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst setup-dbt-project
Create a dbt project implementing this contract and mapping. Simulate seed data matching the schemas if no source is attached. Above every CTE, comment which STTM rule it implements. Then tell me exactly how to run it and where the tables will appear.

/ai-catalyst generate-dag
An Airflow DAG for this product: schedule from the delivery commitment, a source-freshness check first, then the dbt build, then a publish step. Explain every line.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/setup-dbt-project ... and /generate-dag ... with the same files attached</pre></div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 10 min &nbsp;·&nbsp; back at 14:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Debrief — in this order

**🖊️ Whiteboard — debrief, in this order**

- What is dbt?
- What is in the YAML?
- OLTP vs OLAP?
- Which commands?
- And Airflow?
- How well did contracts translate?

![What dbt does with a SELECT, and the three layers](../common/diagrams/png/d2-dbt-flow.png)

- Every model is a SELECT; dbt writes the CREATE; the warehouse does the work.
- A test is a query for rows that must never exist; zero rows is a pass.
- Staging renames and casts, one model per source; intermediate joins; marts are consumer-aligned.

### Activity 8b — review another pair's generated code

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 8b &nbsp;—&nbsp; Review another pair's generated code &nbsp;·&nbsp; Pairs, swapped &nbsp;·&nbsp; 20 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Exchange your contract and one small piece of generated dbt or Airflow code with another pair. Spend five minutes explaining the intended behaviour, ten reviewing, five discussing. As the reviewer: trace one requirement or mapping rule into the code; find one assumption about keys, nulls, dates or scheduling; suggest a concrete check and say what should happen if it fails.</strong><br><br><strong>Produce:</strong><br>• The line(s) that implement one rule<br>• One assumption the code makes that the contract does not state<br>• One check, and the action on failure<br><br><strong>Debrief questions:</strong><br>• Did the code and the contract disagree anywhere?<br>• Which assumption would have reached production unnoticed?</div></div>

### Close — corrections and closing discussion

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ The revised mapping and commitments, the generated dbt and Airflow project, the commands that worked and the unresolved setup issues, saved per pair.<br>☐ Each learner can locate one output table and explain one transformation in it.<br>☐ Access problems recorded separately from problems in the generated code, with an owner for each.<br>☐ Anyone still blocked on the dev container or Databricks has a platform-support contact for tonight.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>